# 1. ETL RaSa Multidimensional

**Autores**

* DIANA CAROLINA LEAL MARTINEZ
* DIEGO ALEJANDRO FIGUEREDO SALAMANCA
* MARIA CAMILA REYES GIL
* SILVIA CARDENAS SANTOS

## 1.1. Configuración, conexión y funciones

Se utiliza la configuración de conexión de para Spark en Colab


In [ ]:
# Configuración servidor base de datos
db_user = 'DB_202615_'
db_psswd = ''
db_host = '157.253.236.120'
db_port = 8080
# base de datos destino del modelo multidimensional
db_destino = 'DB_202615_'      # su base de datos personal

if not db_user or not db_psswd or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'

# origen de extracción de datos
source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/WWImportersTransactional?{jdbc_opciones}'
dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

In [8]:
import os
import sys

# En la máquina virtual ya está todo instalado
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'

if IN_COLAB or IN_CODESPACES:
    %pip install -q "pyspark==4.0.3" "pandas==2.2.3" "numpy>=1.26,<2.5"

if IN_COLAB:
    # Spark 4 requiere Java 17 o superior
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f'Entorno detectado: {ENVIRONMENT}')

from pyspark.sql import functions as f, SparkSession, types as t
from pyspark.sql.functions import udf, col, length, isnan, when, count, regexp_replace
from datetime import datetime

from pathlib import Path
import urllib.request

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()
print(f'Conector JDBC: {MYSQL_JAR}')

Entorno detectado: colab
Conector JDBC: /root/.miid4103/jars/mysql-connector-j-9.7.0.jar


In [9]:
import os
import sys
from pyspark.sql import SparkSession

# Usar el mismo Python del entorno de Colab
os.environ['PYSPARK_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('ETL_RaSa_Multidimensional')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.jars', MYSQL_JAR_URI)
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.enabled', 'false')
    .config('spark.ui.showConsoleProgress', 'false')
    .config('spark.sql.ansi.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

print(f'Spark iniciado: {spark.version}')
spark.range(3).show()

Spark iniciado: 4.0.3
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



In [10]:
# otras librerias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, sequence, to_date, year, month, dayofmonth, date_format, dayofweek
from pyspark.sql import functions as sf
from pyspark.sql import Window
from pyspark.sql.functions import row_number, trim
import re
from pyspark.sql.functions import col, abs

In [11]:
def obterner_dataframe_desde_csv(_PATH, _sep):
    return spark.read.load(_PATH, format="csv", sep=_sep, inferSchema="true", header='true')

def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

## guardar_db
def guardar_db(db_connection_string, df, tabla, db_user, db_psswd, modo='overwrite'):
    # Con modo='overwrite', truncate vacía la tabla y la vuelve a llenar sin cambiar su estructura
    df.select('*').write.format('jdbc') \
      .mode(modo) \
      .option('truncate', 'true') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 10000) \
      .option('numPartitions', 4) \
      .save()

In [12]:
def extraer_anio(columna):
    return sf.regexp_extract(sf.col(columna).cast('string'), r'(\d{4})', 1).cast('int')


# 2. Dimensionnes — Fecha, Condición de pago y Niveles de servicio

## 2.1. Dim Fecha

### 2.1.1. Extraer Fechas

Tal como se relato en diseño de servios se traen las bases de datos 
- FuenteAreasDeServicio_Copia_E (F1)
- FuenteTiposBeneficio_Copia_E (F2)
- FuentePlanesBeneficio_Copia_E (F3)

### 2.1.2. Fuente de area de servicios

In [13]:
sql_fuente_areas_servicios= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.FuenteAreasDeServicio_ETL
order by
	IdAreaDeServicio_T asc) as area_servivio
"""
funte_areas_servio= obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_areas_servicios, db_user, db_psswd)
funte_areas_servio.show(10)

+------------------+--------------------+-------------+--------------------+--------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|             Condado|  Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+--------------------+--------+------------+-----+--------+-----+
|             12018|NoVa10207VA038000...|        51510|     Alexandria City|Virginia|      158309| NULL|    NULL| 2018|
|             12018|NoVa10207VA038000...|        51610|   Falls Church City|Virginia|       14309| NULL|    NULL| 2018|
|             12018|NoVa10207VA038000...|        51107|      Loudoun County|Virginia|      427592|520.0|   822.0| 2018|
|             12018|NoVa10207VA038000...|        51013|    Arlington County|Virginia|      232965| 26.0|  8960.0| 2018|
|             12018|NoVa10207VA038000...|        51059|      Fairfax County|Virginia|     1139720|396.0|  2878.0| 2018|
|             12018|NoVa10207VA038000...

### 2.1.3. Fuente tipos de beneficios

In [14]:
sql_fuente_tipos_beneficios= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.FuenteTiposBeneficio_ETL
ORDER BY IdTipoBeneficio_T ASC) as tipos_beneficios
"""
funte_tipos_beneficios= obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_tipos_beneficios, db_user, db_psswd)
funte_tipos_beneficios.show(10)

+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|    UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|                5|Abortion For Whic...|                   |   No|                   No|                     No|                                      No|                                     No| 2017|
|                5|Abortion For Whic...|                   |   No|                  Yes|                     No|                                      No|                                    Yes| 2020|


### 2.1.4. Fuente de planes de beneficio

In [15]:
sql_fuente_planes_beneficios= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.FuentePlanesBeneficio_ETL
ORDER BY IdTipoBeneficio_T ASC) as planes_beneficios
"""
funte_planes_beneficios= obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_planes_beneficios, db_user, db_psswd)
funte_planes_beneficios.show(10)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|                5|         135454040|                      238|                         27|                1|20129IL0340071-00|2021-12-31|        20129|        850|           50|          NULL|
|                5|         216714036|                      238|                         45|                3|90714MS0010005-04|2017-12-31|        90714|          0|            0|          NULL|
|                5|      

### 2.1.5. Tranformar Fechas

1. Se selecciona la fecha para obtener los rangos
   * areas de servicio

In [16]:
funte_areas_servio.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+---------+---------+
|min_fecha|max_fecha|
+---------+---------+
|     2017|     2021|
+---------+---------+



* Tipos de beneficios

In [17]:
funte_tipos_beneficios.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+---------+---------+
|min_fecha|max_fecha|
+---------+---------+
|     2017|     2021|
+---------+---------+



In [18]:
funte_planes_beneficios.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+----------+-----------+
| min_fecha|  max_fecha|
+----------+-----------+
|2017-12-31|Dec 31,2021|
+----------+-----------+



2. Derivar Anio de la fecha de F3. Para F1 y F2 construir DATE(Anio,12,31). Usar el corte anual Anio-12-31 para ambos hechos. Registrar cualquier fecha de F3 distinta del corte para validación.

* Se identifico que hay más de un formato por lo que primero se ajusta el formato

In [19]:
regex = r"[0-2]\d{3}-(0[1-9]|1[0-2])-(0[1-9]|[1-2][0-9]|3[0-1])"
funte_planes_beneficios.select("Fecha").filter(funte_planes_beneficios["Fecha"].rlike(regex)).count()

61523

In [20]:
funte_planes_beneficios.select("Fecha").filter(~funte_planes_beneficios["Fecha"].rlike(regex)).count()

9229

> 61523 registros cumplen con el formato y 9229 no cumplen con el formato

In [21]:
regex_texto = r"^[A-Za-z]{3}\s+\d{1,2},\d{4}$"
funte_planes_beneficios.filter(funte_planes_beneficios["Fecha"].rlike(regex_texto)).count()

9229

> De los cumplen, todos tienen el formato tienen mes dia, año.

* se ajuata el formato

In [22]:
funte_planes_beneficios_limpio = funte_planes_beneficios.withColumn(
    "Fecha_Estandarizada",
    sf.when(
        sf.col("Fecha").rlike(regex), 
        sf.to_date(sf.substring(sf.col("Fecha"), 1, 10), "yyyy-MM-dd"))
    .when(
        sf.col("Fecha").rlike(regex_texto), 
        sf.to_date(sf.col("Fecha"), "MMM d,yyyy"))
    .otherwise(None))

In [23]:
funte_planes_beneficios_limpio.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+-------------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|Fecha_Estandarizada|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+-------------------+
|                5|         135454040|                      238|                         27|                1|20129IL0340071-00|2021-12-31|        20129|        850|           50|          NULL|         2021-12-31|
|                5|         216714036|                      238|                         45|                3|90714MS0010005-04|2017-12-31| 

* Elimianar la columan fecha y cambiar el nombre de fecha estandarizad con fecha

In [24]:
funte_planes_beneficios_limpio= funte_planes_beneficios_limpio.drop("Fecha").withColumnRenamed("Fecha_Estandarizada", "Fecha")
funte_planes_beneficios_limpio.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+-------------+-----------+-------------+--------------+----------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|     Fecha|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+-------------+-----------+-------------+--------------+----------+
|                5|         135454040|                      238|                         27|                1|20129IL0340071-00|        20129|        850|           50|          NULL|2021-12-31|
|                5|         216714036|                      238|                         45|                3|90714MS0010005-04|        90714|          0|            0|          NULL|2017-12-31|
|                5|      

* Se identimica de nuevo el rango

In [25]:
funte_planes_beneficios_limpio.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+----------+----------+
| min_fecha| max_fecha|
+----------+----------+
|2017-12-31|2021-12-31|
+----------+----------+



* Con los rangos se crea una tabla de fecha.

In [26]:
fecha_inicio = "2017-12-31"
print(fecha_inicio)
fecha_fin = "2021-12-31"
print(fecha_fin)

2017-12-31
2021-12-31


In [27]:
df_rango = spark.sql(f"""
    SELECT EXPLODE(sequence(
        CAST('{fecha_inicio}' AS DATE), 
        CAST('{fecha_fin}' AS DATE), 
        INTERVAL 1 DAY
    )) as Fecha""")

df_rango.show()

+----------+
|     Fecha|
+----------+
|2017-12-31|
|2018-01-01|
|2018-01-02|
|2018-01-03|
|2018-01-04|
|2018-01-05|
|2018-01-06|
|2018-01-07|
|2018-01-08|
|2018-01-09|
|2018-01-10|
|2018-01-11|
|2018-01-12|
|2018-01-13|
|2018-01-14|
|2018-01-15|
|2018-01-16|
|2018-01-17|
|2018-01-18|
|2018-01-19|
+----------+
only showing top 20 rows


* se extrae el Dia, Mes, el año.

In [28]:
Tabla_Fecha = df_rango.selectExpr(
    "Fecha as Fecha",
    "day(Fecha) as Dia",
    "month(Fecha) as Mes",
    "year(Fecha) as anioa"
)  
Tabla_Fecha.show(10)

+----------+---+---+-----+
|     Fecha|Dia|Mes|anioa|
+----------+---+---+-----+
|2017-12-31| 31| 12| 2017|
|2018-01-01|  1|  1| 2018|
|2018-01-02|  2|  1| 2018|
|2018-01-03|  3|  1| 2018|
|2018-01-04|  4|  1| 2018|
|2018-01-05|  5|  1| 2018|
|2018-01-06|  6|  1| 2018|
|2018-01-07|  7|  1| 2018|
|2018-01-08|  8|  1| 2018|
|2018-01-09|  9|  1| 2018|
+----------+---+---+-----+
only showing top 10 rows


3. Generar Id_Fecha como entero AAAAMMDD y Fecha como DATE, visualizada AAAA-MM-DD. Eliminar repeticiones de año.

In [29]:
Tabla_Fecha_con_id = Tabla_Fecha.withColumn("ID_Fecha_T",regexp_replace(col("Fecha"), "-", ""))
Tabla_Fecha_con_id.show()

+----------+---+---+-----+----------+
|     Fecha|Dia|Mes|anioa|ID_Fecha_T|
+----------+---+---+-----+----------+
|2017-12-31| 31| 12| 2017|  20171231|
|2018-01-01|  1|  1| 2018|  20180101|
|2018-01-02|  2|  1| 2018|  20180102|
|2018-01-03|  3|  1| 2018|  20180103|
|2018-01-04|  4|  1| 2018|  20180104|
|2018-01-05|  5|  1| 2018|  20180105|
|2018-01-06|  6|  1| 2018|  20180106|
|2018-01-07|  7|  1| 2018|  20180107|
|2018-01-08|  8|  1| 2018|  20180108|
|2018-01-09|  9|  1| 2018|  20180109|
|2018-01-10| 10|  1| 2018|  20180110|
|2018-01-11| 11|  1| 2018|  20180111|
|2018-01-12| 12|  1| 2018|  20180112|
|2018-01-13| 13|  1| 2018|  20180113|
|2018-01-14| 14|  1| 2018|  20180114|
|2018-01-15| 15|  1| 2018|  20180115|
|2018-01-16| 16|  1| 2018|  20180116|
|2018-01-17| 17|  1| 2018|  20180117|
|2018-01-18| 18|  1| 2018|  20180118|
|2018-01-19| 19|  1| 2018|  20180119|
+----------+---+---+-----+----------+
only showing top 20 rows


### 2.1.6. Cargar Dim Fecha

Se emplea la función de *"guardar_db"* para cargar la base de datos. Se emplea el sufijo "_ELT" para diferenciar las bases de datos.

In [30]:
guardar_db(dest_db_connection_string, Tabla_Fecha_con_id,f'{db_destino}.Fecha_ELT', db_user, db_psswd)

## 2.2. Dim Codiciones de pago

### 2.2.1. Extraer

#### 2.2.1.1. Fuente condiciones de pago

En este punto, aparte de importar la base, se importan los valores unicos por ID, pues se ha detectado previamente que son duplicados.
> **Regla del negocio:**  Puede que existan todavía duplicados de los registros en las fuentes, por favor revisar y eliminarlos.

In [31]:
sql_condiciones_pago= """
(SELECT DISTINCT
    IdCondicionesDePago_T, Descripcion, Tipo
FROM
    RaSaTransaccional_ETL.FuenteCondicionesDePago_ETL
ORDER BY IdCondicionesDePago_T ASC
) as condiciones_pago
"""
fuente_condiciones_pago= obtener_dataframe_de_bd(source_db_connection_string, sql_condiciones_pago, db_user, db_psswd)
fuente_condiciones_pago.show(10)

+---------------------+--------------------+-----------+
|IdCondicionesDePago_T|         Descripcion|       Tipo|
+---------------------+--------------------+-----------+
|                    9|           No Charge|Coseguridad|
|                   17|Copay per Day aft...|   Copagado|
|                   18|No Charge after d...|   Coseguro|
|                   27|Coinsurance after...|   Coseguro|
|                   34|      Not Applicable|   Copagado|
|                   36|      Not Applicable|   Coseguro|
|                   45|         Coinsurance|   Coseguro|
|                   51|           No Charge|     Copago|
|                   68|Copay per Stay af...|     Copago|
|                   85|Copay per Day bef...|     Copago|
+---------------------+--------------------+-----------+
only showing top 10 rows


### 2.2.2. Tranformar

*  Identificar valores únicos, mapear variantes reconocibles del tipo a Copago o Coseguro. Valores no clasificables se modifican en el condicional.
  > **Regla del negocio:**
> * En la fuente de condiciones de pago se eliminaron los registros con el mismo tipo y descripción que tenía diferentes identificadores.
> * En la fuente de condiciones de pago a. Se actualizaron los registros con tipo NaN b. Sin embargo, siguen existiendo errores de consistencia, se le pide corregir el tipo a solo Copago y Coseguro.

In [32]:
fuente_condiciones_pago.columns

['IdCondicionesDePago_T', 'Descripcion', 'Tipo']

In [33]:
fuente_condiciones_pago.select('Tipo').distinct().show()

+-----------+
|       Tipo|
+-----------+
|Coseguridad|
|     Copago|
|   Copagado|
|   Coseguro|
+-----------+



> tal como idica la norma toda via hay errores.

Se acontinuación se hace un condicional

In [34]:
fuente_condiciones_pago_limpio = fuente_condiciones_pago.withColumn(
    "Tipo_Estandarizada",
    sf.when(sf.col("Tipo") == "Coseguridad", "Coseguro")
    .when(sf.col("Tipo") == "Copagado", "Copago")
    .otherwise(sf.col("Tipo")),
)

In [35]:
fuente_condiciones_pago_limpio.show()

+---------------------+--------------------+-----------+------------------+
|IdCondicionesDePago_T|         Descripcion|       Tipo|Tipo_Estandarizada|
+---------------------+--------------------+-----------+------------------+
|                    9|           No Charge|Coseguridad|          Coseguro|
|                   17|Copay per Day aft...|   Copagado|            Copago|
|                   18|No Charge after d...|   Coseguro|          Coseguro|
|                   27|Coinsurance after...|   Coseguro|          Coseguro|
|                   34|      Not Applicable|   Copagado|            Copago|
|                   36|      Not Applicable|   Coseguro|          Coseguro|
|                   45|         Coinsurance|   Coseguro|          Coseguro|
|                   51|           No Charge|     Copago|            Copago|
|                   68|Copay per Stay af...|     Copago|            Copago|
|                   85|Copay per Day bef...|     Copago|            Copago|
|           

In [36]:
fuente_condiciones_pago_limpio= fuente_condiciones_pago_limpio.drop("Tipo").withColumnRenamed("Tipo_Estandarizada", "Tipo")
fuente_condiciones_pago_limpio.show()

+---------------------+--------------------+--------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|
+---------------------+--------------------+--------+
|                    9|           No Charge|Coseguro|
|                   17|Copay per Day aft...|  Copago|
|                   18|No Charge after d...|Coseguro|
|                   27|Coinsurance after...|Coseguro|
|                   34|      Not Applicable|  Copago|
|                   36|      Not Applicable|Coseguro|
|                   45|         Coinsurance|Coseguro|
|                   51|           No Charge|  Copago|
|                   68|Copay per Stay af...|  Copago|
|                   85|Copay per Day bef...|  Copago|
|                  102|Copay per Stay be...|  Copago|
|                  119|Copay per Day wit...|  Copago|
|                  136|Copay per Stay wi...|  Copago|
|                  153|Copay after deduc...|  Copago|
|                  170|Copay before dedu...|  Copago|
|                  187|Copay

* Asignar una PK

In [37]:
fuente_condiciones_pago_limpio = fuente_condiciones_pago_limpio.coalesce(1).withColumn('IdCondicionesDePago_DWH', f.monotonically_increasing_id() + 1)
fuente_condiciones_pago_limpio.show()

+---------------------+--------------------+--------+-----------------------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|IdCondicionesDePago_DWH|
+---------------------+--------------------+--------+-----------------------+
|                    9|           No Charge|Coseguro|                      1|
|                   17|Copay per Day aft...|  Copago|                      2|
|                   18|No Charge after d...|Coseguro|                      3|
|                   27|Coinsurance after...|Coseguro|                      4|
|                   34|      Not Applicable|  Copago|                      5|
|                   36|      Not Applicable|Coseguro|                      6|
|                   45|         Coinsurance|Coseguro|                      7|
|                   51|           No Charge|  Copago|                      8|
|                   68|Copay per Stay af...|  Copago|                      9|
|                   85|Copay per Day bef...|  Copago|           


### 2.2.3. Cargar Dim Condicones de Pago

In [38]:
guardar_db(dest_db_connection_string, fuente_condiciones_pago_limpio,f'{db_destino}.CondicionDePago_ELT', db_user, db_psswd)

## 2.3. Dim Niveles de servicio

In [39]:
sql_niveles_servicio= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.NivelesDeServicio
) as niveles_servicio
"""
fuente_niveles_servicio= obtener_dataframe_de_bd(source_db_connection_string, sql_niveles_servicio, db_user, db_psswd)
fuente_niveles_servicio.show(10)

+---------------------+-------------------+---------------+
|IdNivelDeServicio_DWH|IdNivelDeServicio_T|    Descripcion|
+---------------------+-------------------+---------------+
|                    1|                  1|        Nivel 1|
|                    2|                  2|        Nivel 2|
|                    3|                  3|Fuera de la red|
+---------------------+-------------------+---------------+



### 2.3.1. Extraer

### 2.3.2. Tranformar

> **Regla de negocio** En la fuente de áreas de servicio
> a. Se eliminaron los registros con diferentes IdGeografia cuyo contenido en los demás atributos era el mismo. b. Se corrigieron los registros con IdAreaDeServicio e IdGeografia nulos. c. Las fechas erróneas fueron actualizadas por las correctas. d. Se le pide corregir los registros con áreas negativas, multiplicando por -1 esos valores. e. Finalmente, los que presentan un error de digitación en la población actual con 3 ceros y un uno al final del número (0001) debe corregirlos y quitarles estos últimos 4 dígitos, de tal forma que si el valor era 27540001 debe quedar 2754.

### 2.3.3. Cargar Dim Niveles de Servicio

In [40]:
guardar_db(dest_db_connection_string, fuente_niveles_servicio,f'{db_destino}.NivelesDe_servicio_ELT', db_user, db_psswd)

# 3. Dimensiones — Geografía, Áreas de servicio y Asociación

## 3.1. Extracción de áreas de servicio
Extracción de los datos transaccionales desde la fuente `FuenteAreasDeServicio_ETL`.

In [41]:
sql_fuente_areas_servicios = """
(SELECT 
    IdAreaDeServicio_T,
    NombreAreaDeServicio,
    IdGeografia_T,
    Condado,
    Estado,
    PoblacionAct,
    Area,
    Densidad,
    Fecha
FROM
    RaSaTransaccional_ETL.FuenteAreasDeServicio_ETL
WHERE IdAreaDeServicio_T IS NOT NULL AND IdGeografia_T IS NOT NULL) as area_servicio
"""

df_areas_servicio_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_areas_servicios, db_user, db_psswd)
df_areas_servicio_raw.show(10)

+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|      464269|805.0|   577.0| 2017|
|         100722019|New Jersey  - Med...|        34023| Middlesex County|New Jersey|      860807|311.0|  2768.0| 2019|
|         100922020|New Jersey - Medi...|        34019| Hunterdon County|New Jersey|      129924|430.0|   302.0| 2020|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|      518117|185.0|  2801.0| 2018|
|         101062020|New Jersey - Medi...|        34031|   Passaic County|New Jersey|      518117|185.0|  2801.0| 2020|
|         101082018|New Jersey - Medi...|       

## 3.2. Transformación y reglas de negocio
Aplicación de reglas de negocio de RaSA:
- Corrección de áreas negativas multiplicando por `-1`.
- Corrección del error de digitación en población actual eliminando los últimos 4 dígitos (`0001`).
- Construcción de las dimensiones `Geografia`, `AreasDeServicio` y la tabla de asociación `AsociacionAreaServicioGeografia`.

In [42]:
from pyspark.sql import functions as sf

# 1. Limpieza y aplicación de reglas de negocio
df_as_limpio = df_areas_servicio_raw \
    .withColumn("Area_Corregido", sf.when(sf.col("Area") < 0, sf.col("Area") * -1).otherwise(sf.col("Area"))) \
    .withColumn(
        "PoblacionAct_Corregido",
        sf.when(sf.col("PoblacionAct").cast("string").rlike("0001$"), 
                sf.regexp_replace(sf.col("PoblacionAct").cast("string"), "0001$", "").cast("long"))
        .otherwise(sf.col("PoblacionAct"))
    )

# 2. Dimensión GEOGRAFIA
df_geografia_base = df_as_limpio.select(
    "IdGeografia_T", "Estado", "Condado", "PoblacionAct_Corregido", "Area_Corregido", "Densidad"
).dropDuplicates(["IdGeografia_T"])

df_geografia = df_geografia_base.coalesce(1).withColumn("IdGeografia_DWH", sf.monotonically_increasing_id() + 1)

df_geografia_final = df_geografia.select(
    "IdGeografia_DWH",
    "IdGeografia_T",
    sf.col("Estado").alias("Estado"),
    sf.col("Condado").alias("Condado"),
    sf.col("Area_Corregido").alias("AreaAct"),
    sf.col("Densidad").alias("DensidadAct"),
    sf.col("PoblacionAct_Corregido").alias("PoblacionAct")
)

# 3. Dimensión AREAS DE SERVICIO
df_areas_base = df_as_limpio.select(
    "IdAreaDeServicio_T", 
    sf.col("NombreAreaDeServicio").alias("Nombre"), 
    sf.col("Fecha").alias("AnnioCreacion")
).dropDuplicates(["IdAreaDeServicio_T"])

df_areas_servicio = df_areas_base.coalesce(1).withColumn("IdAreaDeServicio_DWH", sf.monotonically_increasing_id() + 1)

df_areas_servicio_final = df_areas_servicio.select(
    "IdAreaDeServicio_DWH",
    "IdAreaDeServicio_T",
    "Nombre",
    "AnnioCreacion"
)

# 4. Tabla de Asociación (AsociacionAreaServicioGeografia)
df_asociacion = df_as_limpio.join(
    df_areas_servicio_final, df_as_limpio["IdAreaDeServicio_T"] == df_areas_servicio_final["IdAreaDeServicio_T"], "inner"
).join(
    df_geografia_final, df_as_limpio["IdGeografia_T"] == df_geografia_final["IdGeografia_T"], "inner"
).select(
    "IdAreaDeServicio_DWH",
    "IdGeografia_DWH"
).dropDuplicates()

## 3.3. Carga de datos a las Dimensiones
Persistencia de los resultados utilizando el sufijo `_ELT`.

In [43]:
# Cargar Geografía
guardar_db(dest_db_connection_string, df_geografia_final, f'{db_destino}.Geografia_ELT', db_user, db_psswd)

# Cargar Áreas de Servicio
guardar_db(dest_db_connection_string, df_areas_servicio_final, f'{db_destino}.AreasDeServicio_ELT', db_user, db_psswd)

# Cargar Asociación
guardar_db(dest_db_connection_string, df_asociacion, f'{db_destino}.AsociacionAreaServicioGeografia_ELT', db_user, db_psswd)

print("¡Proceso ETL de Geografía, Áreas de Servicio y Asociación finalizado y cargado con éxito!")

¡Proceso ETL de Geografía, Áreas de Servicio y Asociación finalizado y cargado con éxito!


# 4. Dimensiones — Tipos de beneficio, Minicondiciones y Proveedor

* ETL de TiposBeneficio, MiniCondicionesTipoBeneficio y Proveedor

Este notebook implementa la extracción, transformación y carga de tres tablas del modelo multidimensional de RaSA:

| Tabla destino | Fuente | Columnas | Papel en el modelo |
| :--- | :--- | :--- | :--- |
| `TiposBeneficio_ELT` | F2 `FuenteTiposBeneficio_ETL` | `IdTipoBeneficio_DWH`, `IdTipoBeneficio_T`, `Nombre` | Dimensión de los tipos de beneficio |
| `MiniCondicionesTipoBeneficio_ELT` | F2 `FuenteTiposBeneficio_ETL` | `IdCondicionesBeneficios_DWH`, las cinco condiciones Yes/No y `UnidadDelLimite` | Minidimensión (historia tipo 4) de las condiciones que cambian de un año a otro |
| `Proveedor_ELT` | F3 `FuentePlanesBeneficio_ETL` | `IdProveedor_DWH`, `IdProveedor_T` | Dimensión de proveedores, solo con identificadores (nota 2 del modelo) |

**Proceso de implementación**

1. Extraer F2 completa y de F3 solo las columnas que se necesitan (`IdProveedor_T`, `IdPlan_T`, `IdTipoBeneficio_T` y `Fecha`).
2. Transformar aplicando las reglas de negocio de RaSA:
   * eliminar duplicados y espacios sobrantes.
   * homologar las condiciones para que solo queden `Yes` y `No`.
   * reemplazar la unidad del límite vacía por `No aplica`.
   * unificar las unidades del límite que solo difieren en mayúsculas y minúsculas.
   * validar que exista una sola combinación de condiciones por tipo de beneficio y año.
   * completar el proveedor faltante con los 5 primeros caracteres de `IdPlan_T`.
3. Asignar las llaves sustitutas (`_DWH`) con `row_number()` ordenado por la llave natural. Así son consecutivas desde 1 y cada ejecución produce las mismas llaves.
4. Cargar en la base personal con el sufijo `_ELT`, reemplazando el contenido anterior, y verificar cuántas filas quedaron en cada tabla.

Manejo de historia (tipo 4). Las condiciones de un tipo de beneficio pueden cambiar de un año a otro, por eso no se guardan en `TiposBeneficio` sino en la minidimensión `MiniCondicionesTipoBeneficio`, que tiene **una fila por cada combinación distinta de condiciones** y no una por beneficio. La factless `HechoHistCondicionesTiposBeneficio` registra qué combinación tuvo cada beneficio en cada periodo y se carga junto con las tablas de hechos, para esas cargas este notebook construye el mapa `(IdTipoBeneficio_T, Anio) → IdCondicionesTipoBeneficio_DWH` y lo valida contra F3 (sección 3.7).

### 4.1. Funciones
`extraer_anio` toma el año de cuatro dígitos de una fecha, sin importar el formato en que venga (`2018`, `2018-12-31` o `Dec 31,2018`). `guardar_db` recibe el modo de escritura, con `overwrite` vacía la tabla y la vuelve a llenar sin cambiar su estructura.

## 4.2. Extracción

### 4.2.1. Fuente de tipos de beneficio (F2)
Se extraen todas las columnas de F2. Los duplicados se eliminan en la transformación para poder reportar cuántos había.

In [44]:
sql_fuente_tipos_beneficio = '''
(SELECT
    IdTipoBeneficio_T,
    Nombre,
    UnidadDelLimite,
    EsEHB,
    EstaCubiertaPorSeguro,
    TieneLimiteCuantitativo,
    ExcluidoDelDesembolsoMaximoDentroDeLaRed,
    ExcluidoDelDesembolsoMaximoFueraDeLaRed,
    Fecha
FROM
    RaSaTransaccional_ETL.FuenteTiposBeneficio_ETL) AS tipos_beneficio
'''
df_tipos_beneficio_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_tipos_beneficio, db_user, db_psswd)
print(f'Registros extraídos de F2: {df_tipos_beneficio_raw.count():,}')
df_tipos_beneficio_raw.show(10)

Registros extraídos de F2: 449
+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|    UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|                5|Abortion For Whic...|                   |   No|                   No|                     No|                                      No|                                     No| 2017|
|                5|Abortion For Whic...|                   |   No|                  Yes|                     No|                                      No|                

### 4.2.2. Fuente de planes de beneficio (F3)
De F3 solo se necesitan el proveedor y el plan, para construir `Proveedor` y el tipo de beneficio y la fecha, para validar el mapa de condiciones que usarán las tablas de hechos.

In [45]:
sql_fuente_planes_beneficio = '''
(SELECT
    IdProveedor_T,
    IdPlan_T,
    IdTipoBeneficio_T,
    Fecha
FROM
    RaSaTransaccional_ETL.FuentePlanesBeneficio_ETL) AS planes_beneficio
'''
df_planes_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_planes_beneficio, db_user, db_psswd)
print(f'Registros extraídos de F3: {df_planes_raw.count():,}')
df_planes_raw.show(10)

Registros extraídos de F3: 70,752
+-------------+-----------------+-----------------+----------+
|IdProveedor_T|         IdPlan_T|IdTipoBeneficio_T|     Fecha|
+-------------+-----------------+-----------------+----------+
|        16842|16842FL0070128-03|              640|2017-12-31|
|        14002|14002TN0400104-06|              335|2019-12-31|
|        19722|19722NM0010001-02|              885|2020-12-31|
|        81413|81413WI0460015-01|               10|2019-12-31|
|        52697|52697WI0010002-00|              190|2017-12-31|
|        28162|28162OH0060061-04|              810|2019-12-31|
|        20129|20129IL0330060-05|               10|2019-12-31|
|        40572|40572FL0070006-02|              440|2021-12-31|
|        38166|38166WI0180012-06|              620|2017-12-31|
|        70893|70893GA0010009-04|              815|2017-12-31|
+-------------+-----------------+-----------------+----------+
only showing top 10 rows


## 4.3. Transformación
### 4.3.1. Limpieza de F2
**Regla de negocio:** Puede que existan todavía duplicados de los registros en las fuentes, por favor revisar y eliminarlos.

Primero se quitan los espacios sobrantes de los textos, para que dos filas que solo difieren en espacios cuenten como duplicadas. El año de `Fecha` se guarda como entero en `Anio`.

In [46]:
CONDICIONES_YES_NO = ['EsEHB', 'EstaCubiertaPorSeguro', 'TieneLimiteCuantitativo',
                      'ExcluidoDelDesembolsoMaximoDentroDeLaRed', 'ExcluidoDelDesembolsoMaximoFueraDeLaRed']
ATRIBUTOS_CONDICION = CONDICIONES_YES_NO + ['UnidadDelLimite']

df_tb = df_tipos_beneficio_raw.withColumn('Nombre', sf.regexp_replace(sf.trim(sf.col('Nombre')), r'\s+', ' '))
for c in ATRIBUTOS_CONDICION:
    df_tb = df_tb.withColumn(c, sf.trim(sf.col(c)))
df_tb = df_tb.withColumn('Anio', extraer_anio('Fecha')).drop('Fecha')

filas_antes = df_tb.count()
df_tb = df_tb.dropDuplicates()
filas_despues = df_tb.count()
print(f'Filas de F2: {filas_antes:,} | sin duplicados: {filas_despues:,} | duplicados eliminados: {filas_antes - filas_despues:,}')
print(f'Años en F2: {sorted(r.Anio for r in df_tb.select("Anio").distinct().collect() if r.Anio is not None)}')
print(f'Filas sin año: {df_tb.filter(sf.col("Anio").isNull()).count():,}')

Filas de F2: 449 | sin duplicados: 345 | duplicados eliminados: 104
Años en F2: [2017, 2018, 2019, 2020, 2021]
Filas sin año: 0


### 4.3.2. Homologación de las condiciones a `Yes` y `No`
**Regla de negocio:** En la fuente de tipos de beneficio siguen existiendo errores de consistencia en los valores de las condiciones, se le pide cambiar los diferentes valores para que queden únicamente "Yes" y "No".

Primero se revisan los valores que trae cada condición. En el entendimiento de datos se encontraron variantes como `Si`, `True`, `Nein` y `False`.

In [47]:
def resumen_valores(df, columnas):
    # Cuenta las filas de cada valor de cada columna, en una sola tabla
    pares = sf.explode(sf.array(*[
        sf.struct(sf.lit(c).alias('Columna'), sf.col(c).cast('string').alias('Valor')) for c in columnas
    ]))
    return (df.select(pares.alias('par'))
              .select('par.Columna', 'par.Valor')
              .groupBy('Columna', 'Valor').count()
              .orderBy('Columna', 'Valor'))

print('Valores de las condiciones antes de homologar:')
resumen_valores(df_tb, CONDICIONES_YES_NO).show(60, truncate=False)

Valores de las condiciones antes de homologar:
+----------------------------------------+-----+-----+
|Columna                                 |Valor|count|
+----------------------------------------+-----+-----+
|EsEHB                                   |No   |225  |
|EsEHB                                   |True |17   |
|EsEHB                                   |Yes  |103  |
|EstaCubiertaPorSeguro                   |False|2    |
|EstaCubiertaPorSeguro                   |No   |30   |
|EstaCubiertaPorSeguro                   |Yes  |313  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|No   |327  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|Yes  |18   |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |No   |236  |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |Yes  |109  |
|TieneLimiteCuantitativo                 |No   |216  |
|TieneLimiteCuantitativo                 |Si   |10   |
|TieneLimiteCuantitativo                 |Yes  |119  |
+----------------------------------------+-----+-----+



Cada valor se pasa a mayúsculas y se busca en una lista de variantes afirmativas y otra de negativas, si aparece un valor que no está en ninguna de las dos, la celda se detiene y lo muestra. Así se puede agregar a la lista que corresponda en vez de cargarlo sin homologar.

In [48]:
VARIANTES_YES = ['YES', 'Y', 'SI', 'SÍ', 'S', 'TRUE', 'VERDADERO', 'JA', '1']
VARIANTES_NO = ['NO', 'N', 'NEIN', 'FALSE', 'FALSO', '0']

def homologar_yes_no(columna):
    valor = sf.upper(sf.col(columna))
    return (sf.when(valor.isin(VARIANTES_YES), 'Yes')
              .when(valor.isin(VARIANTES_NO), 'No')
              .otherwise(sf.col(columna)))

for c in CONDICIONES_YES_NO:
    df_tb = df_tb.withColumn(c, homologar_yes_no(c))

resumen_despues = resumen_valores(df_tb, CONDICIONES_YES_NO)
print('Valores de las condiciones después de homologar:')
resumen_despues.show(60, truncate=False)

sin_homologar = resumen_despues.filter(sf.col('Valor').isNull() | ~sf.col('Valor').isin('Yes', 'No'))
if sin_homologar.count() > 0:
    sin_homologar.show(truncate=False)
    raise ValueError('Hay condiciones con valores distintos de Yes/No: agréguelos a VARIANTES_YES o VARIANTES_NO')
print('Todas las condiciones quedaron con valor Yes o No')

Valores de las condiciones después de homologar:
+----------------------------------------+-----+-----+
|Columna                                 |Valor|count|
+----------------------------------------+-----+-----+
|EsEHB                                   |No   |225  |
|EsEHB                                   |Yes  |120  |
|EstaCubiertaPorSeguro                   |No   |32   |
|EstaCubiertaPorSeguro                   |Yes  |313  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|No   |327  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|Yes  |18   |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |No   |236  |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |Yes  |109  |
|TieneLimiteCuantitativo                 |No   |216  |
|TieneLimiteCuantitativo                 |Yes  |129  |
+----------------------------------------+-----+-----+

Todas las condiciones quedaron con valor Yes o No


### 4.3.3. Unidad del límite
Cuando el beneficio no tiene límite cuantitativo, `UnidadDelLimite` llega vacía. Se reemplaza por `No aplica` para no cargar vacíos en la dimensión. Además, se unifican las unidades que solo difieren en mayúsculas y minúsculas (por ejemplo `Procedure(s) per lifetime` y `Procedure(s) per Lifetime`), conservando la escritura más frecuente.

También se cuentan los casos incoherentes: con límite pero sin unidad o sin límite pero con unidad. En la ejecución no hubo beneficios con límite y sin unidad, pero sí 16 registros sin límite y con unidad. No se modifican porque no es posible saber si el error está en la condición o en la unidad, y se reportan a RaSA.

In [49]:
unidad_vacia = sf.col('UnidadDelLimite').isNull() | (sf.col('UnidadDelLimite') == '')
df_tb = df_tb.withColumn('UnidadDelLimite', sf.when(unidad_vacia, 'No aplica').otherwise(sf.col('UnidadDelLimite')))

# Unidades que solo difieren en mayúsculas/minúsculas: se conserva la escritura más frecuente
clave_unidad = sf.lower(sf.regexp_replace(sf.col('UnidadDelLimite'), r'\s+', ' '))
variantes_unidad = (df_tb.groupBy('UnidadDelLimite').count()
    .withColumn('clave', clave_unidad)
    .withColumn('orden', sf.row_number().over(
        Window.partitionBy('clave').orderBy(sf.col('count').desc(), sf.col('UnidadDelLimite')))))
unidad_canonica = variantes_unidad.filter(sf.col('orden') == 1).select('clave', sf.col('UnidadDelLimite').alias('UnidadCanonica'))
unidades_unificadas = (variantes_unidad.filter(sf.col('orden') > 1).join(unidad_canonica, 'clave')
                                       .select('UnidadDelLimite', 'UnidadCanonica', 'count'))
print(f'Unidades unificadas por mayúsculas/minúsculas: {unidades_unificadas.count()}')
unidades_unificadas.show(truncate=False)
df_tb = (df_tb.withColumn('clave', clave_unidad)
              .join(unidad_canonica, 'clave', 'left')
              .withColumn('UnidadDelLimite', sf.coalesce('UnidadCanonica', 'UnidadDelLimite'))
              .drop('clave', 'UnidadCanonica'))

df_tb.groupBy('TieneLimiteCuantitativo', 'UnidadDelLimite').count() \
     .orderBy('TieneLimiteCuantitativo', 'UnidadDelLimite').show(50, truncate=False)

con_limite_sin_unidad = df_tb.filter((sf.col('TieneLimiteCuantitativo') == 'Yes') & (sf.col('UnidadDelLimite') == 'No aplica')).count()
sin_limite_con_unidad = df_tb.filter((sf.col('TieneLimiteCuantitativo') == 'No') & (sf.col('UnidadDelLimite') != 'No aplica')).count()
print(f'Con límite cuantitativo pero sin unidad: {con_limite_sin_unidad:,}')
print(f'Sin límite cuantitativo pero con unidad: {sin_limite_con_unidad:,}')

Unidades unificadas por mayúsculas/minúsculas: 1
+-------------------------+-------------------------+-----+
|UnidadDelLimite          |UnidadCanonica           |count|
+-------------------------+-------------------------+-----+
|Procedure(s) per lifetime|Procedure(s) per Lifetime|1    |
+-------------------------+-------------------------+-----+

+-----------------------+-------------------------------+-----+
|TieneLimiteCuantitativo|UnidadDelLimite                |count|
+-----------------------+-------------------------------+-----+
|No                     |Dollars per Year               |1    |
|No                     |Exam(s) per Year               |1    |
|No                     |Item(s) per 3 Years            |1    |
|No                     |Item(s) per Year               |1    |
|No                     |Months per 3 Years             |1    |
|No                     |No aplica                      |200  |
|No                     |Procedure(s) per Episode       |1    |
|No       

### 4.3.4. Una sola combinación de condiciones por tipo de beneficio y año
**Regla de negocio:** En la fuente de tipos de beneficios no deberían existir diferentes condiciones y unidad de límite para el mismo tipo de beneficio en el mismo año. Esto fue arreglado y solo debe existir un valor de condiciones y unidad límite de tipo de beneficio por año.

Esta validación se hace después de homologar, porque antes `Si` y `Yes` contarían como dos valores distintos. Por la misma razón se vuelven a quitar duplicados: dos filas que solo diferían en `Si`/`Yes` ahora son iguales, se espera que no haya ningún par con más de una combinación.

In [50]:
filas_antes = df_tb.count()
df_tb = df_tb.dropDuplicates().cache()
print(f'Filas que quedaron repetidas después de homologar: {filas_antes - df_tb.count():,} | filas finales de F2: {df_tb.count():,}')

conflictos = (df_tb.groupBy('IdTipoBeneficio_T', 'Anio')
                   .agg(sf.countDistinct(*ATRIBUTOS_CONDICION).alias('combinaciones'))
                   .filter(sf.col('combinaciones') > 1))
print(f'Pares (tipo de beneficio, año) con más de una combinación de condiciones: {conflictos.count():,}')
conflictos.orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

Filas que quedaron repetidas después de homologar: 0 | filas finales de F2: 345
Pares (tipo de beneficio, año) con más de una combinación de condiciones: 0
+-----------------+----+-------------+
|IdTipoBeneficio_T|Anio|combinaciones|
+-----------------+----+-------------+
+-----------------+----+-------------+



### 4.3.5. Dimensión TiposBeneficio
La dimensión guarda solo los atributos que no cambian: la llave natural y el nombre. En el entendimiento de datos el nombre fue 1 a 1 con su ID (178 IDs y 178 nombres), si en la fuente actual un ID tuviera más de un nombre, se conserva el del año más reciente y se reporta.

En la ejecución ningún ID tiene más de un nombre, pero hay 205 IDs y 204 nombres distintos, es decir, dos IDs comparten nombre. Se conservan ambos porque son llaves naturales distintas que F3 puede usar, y se muestran para reportarlos a RaSA.

In [51]:
df_nombres = (df_tb.filter(sf.col('IdTipoBeneficio_T').isNotNull())
                   .select('IdTipoBeneficio_T', 'Nombre', 'Anio').distinct())

ids_con_varios_nombres = (df_nombres.groupBy('IdTipoBeneficio_T')
                                    .agg(sf.countDistinct('Nombre').alias('nombres'))
                                    .filter(sf.col('nombres') > 1))
print(f'IDs con más de un nombre: {ids_con_varios_nombres.count()}')
ids_con_varios_nombres.join(df_nombres, 'IdTipoBeneficio_T').orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

nombre_mas_reciente = Window.partitionBy('IdTipoBeneficio_T').orderBy(sf.col('Anio').desc(), sf.col('Nombre'))
df_tipos_beneficio_final = (df_nombres
    .withColumn('orden', sf.row_number().over(nombre_mas_reciente))
    .filter(sf.col('orden') == 1)
    .withColumn('IdTipoBeneficio_DWH', sf.row_number().over(Window.orderBy('IdTipoBeneficio_T')))
    .select('IdTipoBeneficio_DWH', 'IdTipoBeneficio_T', 'Nombre')
    .cache())

print(f'TiposBeneficio: {df_tipos_beneficio_final.count()} filas | '
      f'IDs distintos: {df_tipos_beneficio_final.select("IdTipoBeneficio_T").distinct().count()} | '
      f'nombres distintos: {df_tipos_beneficio_final.select("Nombre").distinct().count()}')
df_tipos_beneficio_final.show(10, truncate=False)

# Nombres que comparten varios IDs (sin distinguir mayúsculas)
nombres_compartidos = (df_tipos_beneficio_final
    .groupBy(sf.lower('Nombre').alias('NombreNormalizado'))
    .agg(sf.count('*').alias('ids'),
         sf.sort_array(sf.collect_list('IdTipoBeneficio_T')).alias('IdsTipoBeneficio'),
         sf.collect_set('Nombre').alias('Nombres'))
    .filter(sf.col('ids') > 1))
print(f'Nombres compartidos por más de un ID: {nombres_compartidos.count()}')
nombres_compartidos.show(truncate=False)

IDs con más de un nombre: 0
+-----------------+-------+------+----+
|IdTipoBeneficio_T|nombres|Nombre|Anio|
+-----------------+-------+------+----+
+-----------------+-------+------+----+

TiposBeneficio: 205 filas | IDs distintos: 205 | nombres distintos: 204
+-------------------+-----------------+-----------------------------------------------+
|IdTipoBeneficio_DWH|IdTipoBeneficio_T|Nombre                                         |
+-------------------+-----------------+-----------------------------------------------+
|1                  |5                |Abortion For Which Public Funding Is Prohibited|
|2                  |10               |Accidental Dental                              |
|3                  |15               |Acupuncture                                    |
|4                  |20               |Adult Frames And Lenses                        |
|5                  |25               |Allergy Injections                             |
|6                  |30            

### 4.3.6. Minidimensión MiniCondicionesTipoBeneficio
Se toman las combinaciones distintas de las seis condiciones. La columna `EstaCubiertaPorSeguro` de la fuente se renombra a `EstaCubiertoPorSeguro`, como aparece en el modelo.

In [52]:
df_mini_final = (df_tb.select(*ATRIBUTOS_CONDICION).distinct()
    .withColumn('IdCondicionesBeneficios_DWH', sf.row_number().over(Window.orderBy(*ATRIBUTOS_CONDICION)))
    .select('IdCondicionesBeneficios_DWH',
            sf.col('EstaCubiertaPorSeguro').alias('EstaCubiertoPorSeguro'),
            'EsEHB',
            'TieneLimiteCuantitativo',
            'ExcluidoDelDesembolsoMaximoDentroDeLaRed',
            'ExcluidoDelDesembolsoMaximoFueraDeLaRed',
            'UnidadDelLimite')
    .cache())

print(f'MiniCondicionesTipoBeneficio: {df_mini_final.count()} combinaciones distintas de condiciones')
df_mini_final.show(50, truncate=False)

MiniCondicionesTipoBeneficio: 96 combinaciones distintas de condiciones
+---------------------------+---------------------+-----+-----------------------+----------------------------------------+---------------------------------------+-------------------------------+
|IdCondicionesBeneficios_DWH|EstaCubiertoPorSeguro|EsEHB|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|UnidadDelLimite                |
+---------------------------+---------------------+-----+-----------------------+----------------------------------------+---------------------------------------+-------------------------------+
|1                          |No                   |No   |No                     |No                                      |No                                     |No aplica                      |
|2                          |No                   |No   |No                     |No                                      |Yes                       

### 4.3.7. Mapa de condiciones para las tablas de hechos
`HechoPlanesTiposBeneficio` y `HechoHistCondicionesTiposBeneficio` necesitan saber qué combinación de condiciones tenía cada tipo de beneficio en cada año, por lo que el mapa se arma cruzando F2 ya homologada con la minidimensión por las seis condiciones, también incluye `IdFecha` con el corte anual que coincide con las fechas de F3 y con `Fecha_ELT`.

El mapa no se carga a la base porque no es una tabla del modelo. Quien cargue los hechos puede reconstruirlo leyendo `MiniCondicionesTipoBeneficio_ELT` y aplicando a F2 las transformaciones 3.1 a 3.4. Se valida que:

1. cada par (tipo de beneficio, año) tenga una sola condición, para que no se dupliquen filas en los hechos.
2. todos los pares (tipo de beneficio, año) que usa F3 encuentren su condición en F2.

En la ejecución, 265 de los 610 pares de F3 no tienen condición en F2 para ese año, porque F2 no trae todos los beneficios en todos los años (por ejemplo, el beneficio 5 aparece en 2017, 2018 y 2020, pero F3 también lo usa en 2019 y 2021). Con el manejo de historia tipo 4 una condición sigue vigente hasta que cambia, por eso se recomienda que la carga de los hechos use la condición del último año anterior disponible. La celda cuenta cuántos pares se recuperan así y muestra los que no tienen ninguna condición anterior, para los que se sugiere un registro desconocido.

In [53]:
df_mapa_condiciones = (df_tb
    .join(df_mini_final.withColumnRenamed('EstaCubiertoPorSeguro', 'EstaCubiertaPorSeguro'),
          on=ATRIBUTOS_CONDICION, how='inner')
    .select('IdTipoBeneficio_T',
            'Anio',
            (sf.col('Anio') * 10000 + 1231).alias('IdFecha'),
            sf.col('IdCondicionesBeneficios_DWH').alias('IdCondicionesTipoBeneficio_DWH'))
    .distinct()
    .cache())
df_mapa_condiciones.orderBy('IdTipoBeneficio_T', 'Anio').show(10)

# Validación 1: una sola condición por par (tipo de beneficio, año)
pares_ambiguos = (df_mapa_condiciones.groupBy('IdTipoBeneficio_T', 'Anio').count()
                                     .filter(sf.col('count') > 1).count())
print(f'Pares (tipo de beneficio, año) con más de una condición: {pares_ambiguos:,}')

# Validación 2: los pares de F3 encuentran su condición
df_pares_f3 = df_planes_raw.select('IdTipoBeneficio_T', extraer_anio('Fecha').alias('Anio')).distinct()
pares_sin_condicion = df_pares_f3.join(df_mapa_condiciones, ['IdTipoBeneficio_T', 'Anio'], 'left_anti')
print(f'Pares (tipo de beneficio, año) en F3: {df_pares_f3.count():,} | sin condición en F2: {pares_sin_condicion.count():,}')
pares_sin_condicion.orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

# Pares sin condición: ¿hay una condición de un año anterior que siga vigente?
anios_con_condicion = df_mapa_condiciones.select('IdTipoBeneficio_T', sf.col('Anio').alias('AnioCondicion'))
vigencia = (pares_sin_condicion.join(anios_con_condicion, 'IdTipoBeneficio_T', 'left')
    .groupBy('IdTipoBeneficio_T', 'Anio')
    .agg(sf.max(sf.when(sf.col('AnioCondicion') < sf.col('Anio'), sf.col('AnioCondicion'))).alias('AnioVigente')))
print(f'  se recuperan con la condición vigente de un año anterior: {vigencia.filter(sf.col("AnioVigente").isNotNull()).count():,}')
print(f'  sin ninguna condición anterior: {vigencia.filter(sf.col("AnioVigente").isNull()).count():,}')
vigencia.filter(sf.col('AnioVigente').isNull()).orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

+-----------------+----+--------+------------------------------+
|IdTipoBeneficio_T|Anio| IdFecha|IdCondicionesTipoBeneficio_DWH|
+-----------------+----+--------+------------------------------+
|                5|2017|20171231|                             1|
|                5|2018|20181231|                             3|
|                5|2020|20201231|                            10|
|               10|2017|20171231|                            61|
|               10|2018|20181231|                            12|
|               10|2019|20191231|                            42|
|               10|2021|20211231|                            63|
|               15|2017|20171231|                             1|
|               15|2018|20181231|                             3|
|               15|2019|20191231|                            24|
+-----------------+----+--------+------------------------------+
only showing top 10 rows
Pares (tipo de beneficio, año) con más de una condición: 0
Pares 

### 4.3.8. Dimensión Proveedor
Por simplificación la dimensión solo tiene identificadores y estos se obtienen de F3. En el entendimiento de datos, `IdProveedor_T` coincidió en el 100 % de las filas con los 5 primeros caracteres de `IdPlan_T`, por eso ese prefijo se usa para completar un proveedor faltante, también se reporta cuántas filas no coinciden.

In [54]:
# El prefijo del plan se convierte al mismo tipo de dato que IdProveedor_T para poder compararlos
tipo_proveedor = df_planes_raw.schema['IdProveedor_T'].dataType
df_prov = (df_planes_raw.select('IdProveedor_T', 'IdPlan_T').dropDuplicates()
           .withColumn('ProveedorDesdePlan', sf.substring(sf.trim(sf.col('IdPlan_T')), 1, 5).cast(tipo_proveedor)))

sin_proveedor = df_prov.filter(sf.col('IdProveedor_T').isNull()).count()
sin_proveedor_ni_plan = df_prov.filter(sf.col('IdProveedor_T').isNull() & sf.col('ProveedorDesdePlan').isNull()).count()
no_coinciden = df_prov.filter(sf.col('IdProveedor_T').isNotNull() & sf.col('ProveedorDesdePlan').isNotNull()
                              & (sf.col('IdProveedor_T') != sf.col('ProveedorDesdePlan'))).count()
print(f'Pares (proveedor, plan) distintos en F3: {df_prov.count():,}')
print(f'  sin IdProveedor_T (se completa con el prefijo de IdPlan_T): {sin_proveedor:,}')
print(f'  sin IdProveedor_T ni IdPlan_T (no se pueden recuperar): {sin_proveedor_ni_plan:,}')
print(f'  con IdProveedor_T distinto del prefijo del plan: {no_coinciden:,}')

df_proveedor_final = (df_prov
    .select(sf.coalesce('IdProveedor_T', 'ProveedorDesdePlan').alias('IdProveedor_T'))
    .filter(sf.col('IdProveedor_T').isNotNull())
    .distinct()
    .withColumn('IdProveedor_DWH', sf.row_number().over(Window.orderBy('IdProveedor_T')))
    .select('IdProveedor_DWH', 'IdProveedor_T')
    .cache())

print(f'Proveedor: {df_proveedor_final.count()} proveedores')
df_proveedor_final.show(10)

Pares (proveedor, plan) distintos en F3: 654
  sin IdProveedor_T (se completa con el prefijo de IdPlan_T): 0
  sin IdProveedor_T ni IdPlan_T (no se pueden recuperar): 0
  con IdProveedor_T distinto del prefijo del plan: 0
Proveedor: 171 proveedores
+---------------+-------------+
|IdProveedor_DWH|IdProveedor_T|
+---------------+-------------+
|              1|        10207|
|              2|        11269|
|              3|        11469|
|              4|        11512|
|              5|        12303|
|              6|        12379|
|              7|        12858|
|              8|        14002|
|              9|        14609|
|             10|        15560|
+---------------+-------------+
only showing top 10 rows


## 4.4. Carga
La carga es en modo `overwrite` con `truncate`: cada ejecución vacía las tres tablas y las vuelve a llenar, así el notebook se puede correr de nuevo sin duplicar filas.

In [55]:
tablas_destino = {
    'TiposBeneficio_ELT': df_tipos_beneficio_final,
    'MiniCondicionesTipoBeneficio_ELT': df_mini_final,
    'Proveedor_ELT': df_proveedor_final,
}

for tabla, df in tablas_destino.items():
    guardar_db(dest_db_connection_string, df, f'{db_destino}.{tabla}', db_user, db_psswd, modo='overwrite')
    print(f'{tabla}: {df.count():,} filas cargadas')

print('¡Proceso ETL de TiposBeneficio, MiniCondicionesTipoBeneficio y Proveedor finalizado!')

TiposBeneficio_ELT: 205 filas cargadas
MiniCondicionesTipoBeneficio_ELT: 96 filas cargadas
Proveedor_ELT: 171 filas cargadas
¡Proceso ETL de TiposBeneficio, MiniCondicionesTipoBeneficio y Proveedor finalizado!


### 4.4.1. Verificación de la carga
Se cuentan las filas de cada tabla en la base y se comparan con las del DataFrame cargado. Ambos conteos deben coincidir.

In [56]:
for tabla, df in tablas_destino.items():
    sql_conteo = f'(SELECT COUNT(*) AS filas FROM {db_destino}.{tabla}) AS conteo'
    filas_bd = obtener_dataframe_de_bd(dest_db_connection_string, sql_conteo, db_user, db_psswd).first()['filas']
    filas_df = df.count()
    estado = 'OK' if filas_bd == filas_df else 'REVISAR'
    print(f'{tabla}: DataFrame {filas_df:,} | base de datos {filas_bd:,} | {estado}')

TiposBeneficio_ELT: DataFrame 205 | base de datos 205 | OK
MiniCondicionesTipoBeneficio_ELT: DataFrame 96 | base de datos 96 | OK
Proveedor_ELT: DataFrame 171 | base de datos 171 | OK


# 5. Tablas de Hechos

## 5.1. Extracción

Para los hechos se usan los datos de `RaSaTransaccional_ETL`, no las tablas con sufijo `Copia`. Se reutilizan las condiciones de F2 ya homologadas y las llaves de las dimensiones cargadas.


In [ ]:
# Parámetros de extracción y tamaño de los lotes.
# 5.1. Configuración y extracción. Ejecutar primero las secciones 1 a 4 (dimensiones)
from pyspark.sql import functions as sf, Window, types as t
from uuid import uuid4

BATCH_SIZE_HECHOS = 5000
PARTICIONES_HECHOS = 2
BD_ORIGEN_HECHOS = 'RaSaTransaccional_ETL'

In [ ]:
# Leer tablas y detener el proceso ante filas inválidas definidas por el negocio o llaves repetidas.
def leer_hechos(tabla, origen=False):
    conexion = source_db_connection_string if origen else dest_db_connection_string
    esquema = BD_ORIGEN_HECHOS if origen else db_destino
    return obtener_dataframe_de_bd(conexion, f'(SELECT * FROM {esquema}.{tabla}) AS lectura_hechos', db_user, db_psswd)

def exigir_sin_filas(df, mensaje):
    if df.limit(1).count():
        df.show(20, truncate=False)
        raise ValueError(mensaje)

def exigir_unico(df, columnas, nombre):
    exigir_sin_filas(df.groupBy(*columnas).count().filter(sf.col('count') > 1),
                    f'{nombre}: llave duplicada {columnas}; revisar dimensiones antes de cargar hechos.')


In [ ]:
# Homologar nombres de columnas y me quedo solo con las columnas necesarias
def canonizar_dimension(df, campos, nombre):
    # Aceptar las denominaciones que ya tiene el notebook sin regenerar sus PK.
    expresiones = []
    for destino, candidatos in campos.items():
        encontrados = [c for c in candidatos if c in df.columns]
        if not encontrados:
            raise ValueError(f'{nombre}: falta {destino}. Columnas disponibles: {df.columns}')
        expresiones.append(sf.col(encontrados[0]).alias(destino))
    return df.select(*expresiones).distinct()
# Función para limpiar y estandarizar texto en columnas de DataFrame
def clave_texto(columna):
    return sf.trim(sf.col(columna).cast('string'))

**Fechas incompletas.** Según el caso, si falta el mes se usa enero y si falta el día se usa el día 01. Las fechas deben poder interpretarse y quedar como `YYYY-MM-DD`; una fecha inválida detiene el proceso.


In [ ]:
# Interpretar fechas y completar mes o día ausentes con 01.
def fecha_fuente(columna):
    # RaSa: mes/día ausentes se completan con 01; no inferir fechas ambiguas.
    texto = sf.trim(sf.col(columna).cast('string'))
    texto = (sf.when(texto.rlike(r'^\d{4}$'), sf.concat(texto, sf.lit('-01-01')))
             .when(texto.rlike(r'^\d{4}-\d{2}$'), sf.concat(texto, sf.lit('-01')))
             .otherwise(texto))
    return sf.coalesce(sf.to_date(sf.try_to_timestamp(texto, sf.lit('yyyy-MM-dd'))),
                       sf.to_date(sf.try_to_timestamp(texto, sf.lit('MMM d,yyyy'))))


### 5.1.1 Lectura de Dimensiones

In [ ]:
# Extraer F3 y reutilizar F2 ya homologada anteriormente en las dimensiones.
df_f3_hechos_raw = leer_hechos('FuentePlanesBeneficio_ETL', origen=True)
# df_tb ya contiene las homologaciones de F2 realizadas anteriormente en las dimensiones.
if 'df_tb' not in globals():
    raise ValueError('Ejecute la transformación de F2 de la sección 4 antes de los hechos.')
df_f2_hechos = df_tb.select('IdTipoBeneficio_T', 'Anio', *ATRIBUTOS_CONDICION).distinct()


In [ ]:
# Leer las llaves persistidas de beneficio, proveedor y área.
dim_beneficio_h = canonizar_dimension(leer_hechos('TiposBeneficio_ELT'), {
    'IdTipoBeneficio_DWH': ['IdTipoBeneficio_DWH'], 'IdTipoBeneficio_T': ['IdTipoBeneficio_T']}, 'Beneficio')
dim_proveedor_h = canonizar_dimension(leer_hechos('Proveedor_ELT'), {
    'IdProveedor_DWH': ['IdProveedor_DWH'], 'IdProveedor_T': ['IdProveedor_T']}, 'Proveedor')
dim_area_original_h = leer_hechos('AreasDeServicio_ELT')
dim_area_h = canonizar_dimension(dim_area_original_h, {
    'IdAreaDeServicio_DWH': ['IdAreaDeServicio_DWH'], 'IdAreaDeServicio_T': ['IdAreaDeServicio_T']}, 'Área')

In [ ]:
# Leer los catálogos de niveles y condiciones de pago.
dim_nivel_h = canonizar_dimension(leer_hechos('NivelesDe_servicio_ELT'), {
    'IdNivelDeServicio_DWH': ['IdNivelDeServicio_DWH', 'IdNivelServicio_DWH', 'IdNivelDeServicio_DWT', 'IdNivelServicio_DWT'],
    'IdNivelServicio_T': ['IdNivelDeServicio_T', 'IdNivelServicio_T']}, 'Nivel')
dim_pago_h = canonizar_dimension(leer_hechos('CondicionDePago_ELT'), {
    'IdCondicionDePago_DWH': ['IdCondicionesDePago_DWH', 'IdCondicionDePago_DWH'],
    'IdCondicionDePago_T': ['IdCondicionesDePago_T', 'IdCondicionDePago_T'], 'Tipo': ['Tipo']}, 'Pago')

In [ ]:
# Leer la minidimensión y el catálogo de fechas.
dim_mini_h = leer_hechos('MiniCondicionesTipoBeneficio_ELT').withColumnRenamed('EstaCubiertoPorSeguro', 'EstaCubiertaPorSeguro').distinct()
dim_fecha_original_h = leer_hechos('Fecha_ELT')
dim_fecha_h = canonizar_dimension(dim_fecha_original_h, {
    'IdFecha': ['ID_Fecha_T', 'IdFecha', 'Id_Fecha'], 'Fecha': ['Fecha']}, 'Fecha')
print(f'F3 extraída: {df_f3_hechos_raw.count():,} filas. Lotes JDBC: {BATCH_SIZE_HECHOS:,}.')

F3 extraída: 70,752 filas. Lotes JDBC: 5,000.


## 5.2. Transformación

In [65]:
# Normalizar identificadores, fechas y proveedor de cada oferta.
# 5.2. Transformación de los dos hechos del modelo tipo 4.
df_incidencias_hechos = None
planes_h = df_f3_hechos_raw
for c in ['IdTipoBeneficio_T', 'IdAreaDeServicio_T', 'IdNivelServicio_T', 'IdPlan_T',
          'IdCondicionDePagoCopago_T', 'IdCondicionDePagoCoseguro_T', 'IdProveedor_T']:
    planes_h = planes_h.withColumn(c, clave_texto(c))
planes_h = planes_h.withColumn('FechaEmision', fecha_fuente('Fecha'))
exigir_sin_filas(planes_h.filter(sf.col('FechaEmision').isNull()), 'F3 contiene fechas inválidas o formatos no reconocidos.')
for campo in ['IdTipoBeneficio_T', 'IdPlan_T', 'IdNivelServicio_T', 'IdAreaDeServicio_T']:
    exigir_sin_filas(planes_h.filter(sf.col(campo).isNull() | (sf.col(campo) == '')), f'F3: {campo} vacío.')
planes_h = (planes_h.withColumn('ProveedorDesdePlan', sf.substring('IdPlan_T', 1, 5))
    .withColumn('IdProveedor_T', sf.when(sf.col('IdProveedor_T') == '', None).otherwise(sf.col('IdProveedor_T'))))
exigir_sin_filas(planes_h.filter(sf.col('IdProveedor_T').isNotNull() &
    (sf.col('IdProveedor_T') != sf.col('ProveedorDesdePlan'))), 'Proveedor explícito distinto del prefijo del plan.')
planes_h = planes_h.withColumn('IdProveedor_T', sf.coalesce('IdProveedor_T', 'ProveedorDesdePlan'))


In [66]:
# Convertir medidas a números y detectar valores no numéricos.
# Convertir medidas sin confundir un texto no numérico con un nulo de origen.
for origen, destino in [('valorCopago', 'ValorCopago'), ('valorCoseguro', 'ValorCoseguro'), ('cantidadLimite', 'CantidadLimite')]:
    # Spark no distingue mayúsculas por defecto: conservar temporalmente el valor raw.
    raw = '_raw_' + destino
    planes_h = planes_h.withColumnRenamed(origen, raw)
    planes_h = planes_h.withColumn(destino, sf.expr(f'try_cast(`{raw}` as decimal(18,4))'))
    exigir_sin_filas(planes_h.filter(sf.col(raw).isNotNull() & sf.col(destino).isNull()), f'Medida no numérica: {origen}.')


**Duplicados.** Se eliminan las ofertas exactamente repetidas. Si una misma fecha, plan, beneficio, nivel y área tiene costos o condiciones diferentes, se detiene la carga para revisar el conflicto.


In [67]:
# Eliminar duplicados exactos y validar el grano de la oferta.
grano_planes_h = ['FechaEmision', 'IdPlan_T', 'IdTipoBeneficio_T', 'IdNivelServicio_T', 'IdAreaDeServicio_T']
campos_planes_h = grano_planes_h + ['IdProveedor_T', 'IdCondicionDePagoCopago_T', 'IdCondicionDePagoCoseguro_T',
                                   'ValorCopago', 'ValorCoseguro', 'CantidadLimite']
antes_h = planes_h.count()
planes_h = planes_h.select(*campos_planes_h).distinct().cache()
print(f'Duplicados exactos de F3 eliminados: {antes_h - planes_h.count():,}')
exigir_unico(planes_h, grano_planes_h, 'F3: costos o condiciones diferentes para la misma oferta')


Duplicados exactos de F3 eliminados: 0


In [68]:
# Comprobar unicidad de las dimensiones antes de los cruces.
for nombre, dim, natural, pk in [
    ('Beneficio', dim_beneficio_h, 'IdTipoBeneficio_T', 'IdTipoBeneficio_DWH'),
    ('Proveedor', dim_proveedor_h, 'IdProveedor_T', 'IdProveedor_DWH'),
    ('Área', dim_area_h, 'IdAreaDeServicio_T', 'IdAreaDeServicio_DWH'),
    ('Nivel', dim_nivel_h, 'IdNivelServicio_T', 'IdNivelDeServicio_DWH'),
    ('Pago', dim_pago_h, 'IdCondicionDePago_T', 'IdCondicionDePago_DWH')]:
    exigir_sin_filas(dim.filter(sf.col(natural).isNull() | sf.col(pk).isNull()), f'{nombre}: llave nula.')
    exigir_unico(dim.withColumn(natural, clave_texto(natural)), [natural], nombre)
    exigir_unico(dim, [pk], nombre)
exigir_unico(dim_mini_h, ATRIBUTOS_CONDICION, 'Minicondiciones')
exigir_unico(dim_mini_h, ['IdCondicionesBeneficios_DWH'], 'Minicondiciones')
exigir_sin_filas(dim_mini_h.filter(sf.col('IdCondicionesBeneficios_DWH').isNull()), 'Minicondiciones: PK nula.')
exigir_sin_filas(dim_fecha_h.filter(sf.col('Fecha').isNull() | sf.col('IdFecha').cast('int').isNull() |
    (sf.col('IdFecha').cast('int') != sf.date_format('Fecha', 'yyyyMMdd').cast('int'))), 'Fecha: ID y fecha inconsistentes.')
exigir_unico(dim_fecha_h, ['Fecha'], 'Fecha')
exigir_unico(dim_fecha_h, ['IdFecha'], 'Fecha')


**Condiciones del beneficio.** Las cinco condiciones deben estar homologadas a `Yes` o `No`. Para cada beneficio y año debe existir una sola combinación de condiciones y unidad del límite, como exige el contexto.


In [69]:
# Resolver las combinaciones de F2 contra la minidimensión persistida.
# Resolver F2 contra las PK persistidas; no recalcular identificadores DWH.
f2_h = df_f2_hechos.withColumn('IdTipoBeneficio_T', clave_texto('IdTipoBeneficio_T'))
exigir_sin_filas(f2_h.filter(sf.col('IdTipoBeneficio_T').isNull() | (sf.col('IdTipoBeneficio_T') == '')), 'F2: beneficio vacío.')
exigir_sin_filas(f2_h.filter(sf.col('Anio').isNull() | ~sf.col('Anio').between(1900, 9998)), 'F2: año inválido.')
exigir_unico(f2_h, ['IdTipoBeneficio_T', 'Anio'], 'F2')
mapa_h = f2_h.join(dim_mini_h, ATRIBUTOS_CONDICION, 'left')
exigir_sin_filas(mapa_h.filter(sf.col('IdCondicionesBeneficios_DWH').isNull()), 'F2: combinación sin minidimensión persistida.')


**Historia de condiciones.** El modelo separa el beneficio de sus condiciones mediante una minidimensión y un hecho histórico. Se abre un nuevo periodo cuando cambia la combinación de condiciones; repetir la misma combinación no crea otro cambio.


In [70]:
# Detectar cambios reales y definir el horizonte histórico.
mapa_h = mapa_h.select('IdTipoBeneficio_T', 'Anio', 'IdCondicionesBeneficios_DWH', 'TieneLimiteCuantitativo')
w_h = Window.partitionBy('IdTipoBeneficio_T').orderBy('Anio')
mapa_h = mapa_h.withColumn('_anterior', sf.lag('IdCondicionesBeneficios_DWH').over(w_h))
cambios_h = mapa_h.filter(sf.col('_anterior').isNull() | (sf.col('_anterior') != sf.col('IdCondicionesBeneficios_DWH')))
ultimo_anio_h = max(planes_h.select(sf.max(sf.year('FechaEmision'))).first()[0] or 0,
                    f2_h.select(sf.max('Anio')).first()[0] or 0)
if not ultimo_anio_h:
    raise ValueError('No hay años para construir el horizonte histórico.')
# Foto histórica acotada a los años extraídos: última vigencia al 31/12 del horizonte.
# Una condición sigue vigente aunque no exista una observación en el año siguiente.


**Criterio temporal de esta implementación.** Una condición comienza el 1 de enero del año informado y se mantiene hasta el día anterior al siguiente cambio. El último periodo se cierra el 31 de diciembre del último año extraído. La ausencia de un año en F2 no se interpreta como una baja.


In [ ]:
# Construir intervalos de vigencia sin solapamientos.
intervalos_h = (cambios_h.withColumn('FechaInicio', sf.make_date('Anio', sf.lit(1), sf.lit(1)))
    .withColumn('_siguiente', sf.lead('Anio').over(w_h))
    .withColumn('FechaFin', sf.when(sf.col('_siguiente').isNotNull(),
        sf.date_sub(sf.make_date('_siguiente', sf.lit(1), sf.lit(1)), 1))
        .otherwise(sf.to_date(sf.lit(f'{ultimo_anio_h}-12-31')))))

**Medida `Cambio`.** Según el modelo adjunto, `HechoHistCondicionesTiposBeneficio` es una tabla factless: cada fila registra un periodo y lleva `Cambio = 1`. No representa un costo ni la cantidad de condiciones modificadas.


In [72]:
# Construir el hecho histórico con Cambio igual a 1.
b_h = dim_beneficio_h.withColumn('IdTipoBeneficio_T', clave_texto('IdTipoBeneficio_T'))
historia_h = intervalos_h.join(b_h, 'IdTipoBeneficio_T', 'left')
exigir_sin_filas(historia_h.filter(sf.col('IdTipoBeneficio_DWH').isNull()), 'Historia: beneficio sin dimensión.')
df_hecho_hist_h = historia_h.select('IdTipoBeneficio_DWH',
    sf.col('IdCondicionesBeneficios_DWH').alias('IdCondicionesTipoBeneficio_DWH'),
    sf.date_format('FechaInicio', 'yyyyMMdd').cast('int').alias('IdFechaInicio'),
    sf.date_format('FechaFin', 'yyyyMMdd').cast('int').alias('IdFechaFin'), sf.lit(1).alias('Cambio')).cache()
exigir_unico(df_hecho_hist_h, ['IdTipoBeneficio_DWH', 'IdFechaInicio'], 'Hecho histórico')
exigir_sin_filas(historia_h.filter(sf.col('FechaInicio').isNull() | sf.col('FechaFin').isNull() |
    (sf.col('FechaInicio') > sf.col('FechaFin'))), 'Intervalo histórico inválido.')


**Condiciones vigentes.** Cada oferta recibe las condiciones válidas en su fecha de emisión. No se utiliza una condición futura; si no existe una condición previa disponible, se muestra la incidencia y se detiene la carga.


In [ ]:
# Asignar a cada oferta la condición vigente en su fecha.
# Búsqueda temporal: nunca tomar una condición futura ni cruzar únicamente por año.
ofertas_h = (planes_h.alias('p').join(historia_h.alias('h'),
    (sf.col('p.IdTipoBeneficio_T') == sf.col('h.IdTipoBeneficio_T')) &
    sf.col('p.FechaEmision').between(sf.col('h.FechaInicio'), sf.col('h.FechaFin')), 'left')
    .select('p.*', 'h.IdTipoBeneficio_DWH', 'h.IdCondicionesBeneficios_DWH', 'h.TieneLimiteCuantitativo'))
df_incidencias_hechos = ofertas_h.filter(sf.col('IdCondicionesBeneficios_DWH').isNull())
exigir_sin_filas(df_incidencias_hechos, 'Ofertas sin condición histórica anterior: revisar df_incidencias_hechos. No se inventan condiciones.')

**Área desconocida.** El contexto indica que algunos planes tienen áreas que no existen en F1. Esas ofertas se conservan asignando el identificador `0`, correspondiente a “Desconocido”. Las áreas originales siguen diferenciándose al validar las ofertas.


In [74]:
# Resolver proveedor, nivel y área; conservar áreas ausentes como 0.
for dim, natural, pk in [(dim_proveedor_h, 'IdProveedor_T', 'IdProveedor_DWH'),
                         (dim_nivel_h, 'IdNivelServicio_T', 'IdNivelDeServicio_DWH'),
                         (dim_area_h, 'IdAreaDeServicio_T', 'IdAreaDeServicio_DWH')]:
    ofertas_h = ofertas_h.join(dim.withColumn(natural, clave_texto(natural)), natural, 'left')
    if natural != 'IdAreaDeServicio_T':
        exigir_sin_filas(ofertas_h.filter(sf.col(pk).isNull()), f'Oferta sin correspondencia dimensional: {natural}.')
areas_desconocidas_h = ofertas_h.filter(sf.col('IdAreaDeServicio_DWH').isNull()).count()
ofertas_h = ofertas_h.withColumn('IdAreaDeServicio_DWH', sf.coalesce('IdAreaDeServicio_DWH', sf.lit(0)))


**Copago y coseguro.** Son dos referencias distintas al catálogo de condiciones de pago. Cada una debe corresponder a su tipo correcto: `Copago` o `Coseguro`.


In [ ]:
# Resolver por separado los roles de copago y coseguro.
for rol in ['Copago', 'Coseguro']:
    natural = f'IdCondicionDePago{rol}_T'
    pk = f'IdCondicionDePago{rol}_DWH'
    catalogo = (dim_pago_h.filter(sf.col('Tipo') == rol)
        .select(clave_texto('IdCondicionDePago_T').alias(natural), sf.col('IdCondicionDePago_DWH').alias(pk)))
    ofertas_h = ofertas_h.join(catalogo, natural, 'left')
    exigir_sin_filas(ofertas_h.filter(sf.col(pk).isNull()), f'Condición de {rol} ausente o con tipo incorrecto.')

**Cantidad límite.** Si el beneficio tiene límite cuantitativo y la cantidad es nula o cero, se asigna `333`, como pide el negocio para identificar el error. No se usa `333` para reemplazar textos inválidos o cantidades negativas.

**Valores de pago.** Para 2018, el contexto fija máximos de copago de `3500` y coseguro de `100`. El código también valida copagos no negativos y coseguros entre 0 y 100; los valores fuera de rango se reportan sin recortarlos.


In [76]:
# Aplicar la regla de 333 y validar medidas y cantidad de ofertas.
ofertas_h = ofertas_h.withColumn('CantidadLimite',
    sf.when((sf.col('TieneLimiteCuantitativo') == 'Yes') &
        (sf.col('CantidadLimite').isNull() | (sf.col('CantidadLimite') == 0)), sf.lit(333))
    .otherwise(sf.col('CantidadLimite')))
exigir_sin_filas(ofertas_h.filter((sf.col('ValorCopago') < 0) |
    ~sf.col('ValorCoseguro').between(0, 100) | (sf.col('CantidadLimite') < 0) |
    ((sf.year('FechaEmision') == 2018) & (sf.col('ValorCopago') > 3500))), 'Medidas fuera de los rangos del contexto.')
exigir_unico(ofertas_h, grano_planes_h, 'Ofertas después de búsquedas dimensionales')
if ofertas_h.count() != planes_h.count():
    raise ValueError('Los cruces perdieron o multiplicaron ofertas.')


**Hecho de planes.** Cada fila conserva el plan, la fecha de emisión, el beneficio, el nivel, el área y las condiciones de pago, junto con copago, coseguro y cantidad límite. No se cruza con los condados para evitar repetir estas medidas por cada geografía.


In [ ]:
# Proyectar las columnas del hecho de planes y beneficios.
df_hecho_planes_h = ofertas_h.select(sf.col('IdProveedor_DWH').alias('IdProveedor'),
    sf.date_format('FechaEmision', 'yyyyMMdd').cast('int').alias('IdFechaEmision'),
    sf.col('IdPlan_T').alias('IdPlan'), 'IdAreaDeServicio_DWH', 'IdNivelDeServicio_DWH', 'IdTipoBeneficio_DWH',
    sf.col('IdCondicionesBeneficios_DWH').alias('IdCondicionesTipoBeneficio_DWH'),
    'IdCondicionDePagoCoseguro_DWH', 'IdCondicionDePagoCopago_DWH', 'ValorCoseguro', 'ValorCopago', 'CantidadLimite').cache()

In [78]:
# Preparar las fechas que faltan para las vigencias y emisiones.
# Solo preparar complementos requeridos por los hechos; se escriben en 5.3.
fechas_requeridas_h = (historia_h.select(sf.col('FechaInicio').alias('Fecha'))
    .union(historia_h.select(sf.col('FechaFin').alias('Fecha')))
    .union(planes_h.select(sf.col('FechaEmision').alias('Fecha'))).distinct())
faltantes_fecha_h = fechas_requeridas_h.join(dim_fecha_h.select('Fecha'), 'Fecha', 'left_anti')
expr_fecha_h = {'Fecha': sf.col('Fecha'), 'Dia': sf.dayofmonth('Fecha'), 'Mes': sf.month('Fecha'),
    'anioa': sf.year('Fecha'), 'Anio': sf.year('Fecha'), 'Annio': sf.year('Fecha'),
    'ID_Fecha_T': sf.date_format('Fecha', 'yyyyMMdd'), 'IdFecha': sf.date_format('Fecha', 'yyyyMMdd'),
    'Id_Fecha': sf.date_format('Fecha', 'yyyyMMdd')}
if any(c not in expr_fecha_h for c in dim_fecha_original_h.columns):
    raise ValueError('Fecha tiene columnas adicionales: definir su regla antes de completar fechas.')
df_fechas_adicionales_h = faltantes_fecha_h.select(*[
    expr_fecha_h[c.name].cast(c.dataType).alias(c.name) for c in dim_fecha_original_h.schema])


In [79]:
# Comprobar que el identificador 0 de área esté disponible.
exigir_sin_filas(dim_area_h.filter((sf.col('IdAreaDeServicio_DWH') == 0) &
    (sf.col('IdAreaDeServicio_T').cast('string') != '0')), 'La PK 0 de Área está ocupada por otro miembro.')
exigir_sin_filas(dim_area_h.filter((sf.col('IdAreaDeServicio_T').cast('string') == '0') &
    (sf.col('IdAreaDeServicio_DWH') != 0)), 'La llave natural 0 de Área tiene otra PK.')


In [80]:
# Preparar el miembro desconocido sin duplicarlo.
fila_area_h = {c.name: (0 if c.name in ['IdAreaDeServicio_DWH', 'IdAreaDeServicio_T'] else
    'Desconocido' if c.name == 'Nombre' else None) for c in dim_area_original_h.schema}
df_area_cero_h = spark.range(1).select(*[
    sf.lit(fila_area_h[c.name]).cast(c.dataType).alias(c.name) for c in dim_area_original_h.schema])
df_area_cero_h = df_area_cero_h.join(dim_area_original_h.select('IdAreaDeServicio_DWH'), 'IdAreaDeServicio_DWH', 'left_anti')


In [81]:
# Validar las referencias de fecha de ambos hechos.
fechas_disponibles_h = dim_fecha_h.select(sf.col('IdFecha').cast('int').alias('IdFecha')).union(
    faltantes_fecha_h.select(sf.date_format('Fecha', 'yyyyMMdd').cast('int').alias('IdFecha'))).distinct()
for df, campo in [(df_hecho_planes_h, 'IdFechaEmision'), (df_hecho_hist_h, 'IdFechaInicio'), (df_hecho_hist_h, 'IdFechaFin')]:
    exigir_sin_filas(df.select(sf.col(campo).alias('IdFecha')).join(fechas_disponibles_h, 'IdFecha', 'left_anti'),
                    f'{campo}: fecha sin correspondencia dimensional.')


In [82]:
# Mostrar conteos y muestras de los hechos preparados.
print(f'Planes: {df_hecho_planes_h.count():,}; intervalos históricos: {df_hecho_hist_h.count():,}; ofertas con área 0: {areas_desconocidas_h:,}.')
df_hecho_hist_h.show(5, truncate=False)
df_hecho_planes_h.show(5, truncate=False)


Planes: 70,752; intervalos históricos: 342; ofertas con área 0: 15,309.
+-------------------+------------------------------+-------------+----------+------+
|IdTipoBeneficio_DWH|IdCondicionesTipoBeneficio_DWH|IdFechaInicio|IdFechaFin|Cambio|
+-------------------+------------------------------+-------------+----------+------+
|201                |48                            |20170101     |20211231  |1     |
|203                |21                            |20170101     |20211231  |1     |
|22                 |7                             |20210101     |20211231  |1     |
|24                 |7                             |20170101     |20171231  |1     |
|24                 |15                            |20180101     |20211231  |1     |
+-------------------+------------------------------+-------------+----------+------+
only showing top 5 rows
+-----------+--------------+-----------------+--------------------+---------------------+-------------------+------------------------------

## 5.3. Carga de los Hechos

**Carga por lotes.** Como decisión técnica de la implementación, se envían hasta `5.000` filas por lote JDBC a tablas temporales de preparación. Antes de publicar se comprueba que la cantidad recibida coincida con la preparada.


In [83]:
# Escribir DataFrames en staging mediante batch JDBC.
# 5.3. Lotes JDBC a staging y publicación transaccional de ambos hechos.
def escribir_lotes_hechos(df, tabla):
    (df.coalesce(PARTICIONES_HECHOS).write.format('jdbc').mode('errorifexists')
        .option('url', dest_db_connection_string).option('dbtable', f'{db_destino}.{tabla}')
        .option('user', db_user).option('password', db_psswd)
        .option('driver', 'com.mysql.cj.jdbc.Driver').option('batchsize', BATCH_SIZE_HECHOS)
        .option('numPartitions', PARTICIONES_HECHOS).option('isolationLevel', 'READ_COMMITTED').save())


In [84]:
# Escapar identificadores y construir nombres SQL del esquema destino.
def identificador_h(nombre):
    return '`' + nombre.replace('`', '``') + '`'

def nombre_sql_h(tabla):
    return identificador_h(db_destino) + '.' + identificador_h(tabla)


In [85]:
# Abrir una conexión JDBC con el conector MySQL ya cargado.
def conexion_jdbc_hechos():
    props = spark._jvm.java.util.Properties()
    props.setProperty('user', db_user)
    props.setProperty('password', db_psswd)
    loader = spark._jvm.java.lang.Thread.currentThread().getContextClassLoader()
    driver = loader.loadClass('com.mysql.cj.jdbc.Driver').newInstance()
    return driver.connect(dest_db_connection_string, props)


In [ ]:
# Ejecutar una sentencia SQL y cerrar su recurso.
def ejecutar_sql_h(conexion, sql):
    statement = conexion.createStatement()
    try:
        statement.execute(sql)
    finally:
        statement.close()

In [87]:
# Consultar un conteo SQL y cerrar el resultado y la sentencia.
def escalar_sql_h(conexion, sql):
    statement = conexion.createStatement()
    try:
        result = statement.executeQuery(sql)
        try:
            result.next()
            return result.getLong(1)
        finally:
            result.close()
    finally:
        statement.close()

In [88]:
# Preparar staging y comprobar la cantidad de filas recibidas.
def preparar_staging_hechos(publicaciones, stages, token):
    for i, (destino, df, clave) in enumerate(publicaciones):
        stage = f'_stg_hechos_{token}_{i}'
        stages.append(stage)
        escribir_lotes_hechos(df, stage)
        esperado = df.count()
        conexion = conexion_jdbc_hechos()
        try:
            real = escalar_sql_h(conexion, f'SELECT COUNT(*) FROM {nombre_sql_h(stage)}')
            if real != esperado:
                raise ValueError(f'{stage}: esperado {esperado}, recibido {real}. No se publica.')
            if clave is None:
                ejecutar_sql_h(conexion, f'CREATE TABLE IF NOT EXISTS {nombre_sql_h(destino)} LIKE {nombre_sql_h(stage)}')
        finally:
            conexion.close()
        print(f'Staging {destino}: {real:,} filas verificadas; batchsize={BATCH_SIZE_HECHOS}.')


In [ ]:
# Exigir un motor que permita deshacer la publicación si falla.
def validar_motor_hechos(conexion, publicaciones):
    for destino, _, _ in publicaciones:
        stmt = conexion.prepareStatement('SELECT ENGINE FROM information_schema.TABLES WHERE TABLE_SCHEMA=? AND TABLE_NAME=?')
        try:
            stmt.setString(1, db_destino)
            stmt.setString(2, destino)
            rs = stmt.executeQuery()
            try:
                if not rs.next() or rs.getString(1).upper() != 'INNODB':
                    raise ValueError(f'{destino}: se requiere InnoDB para publicación transaccional.')
            finally:
                rs.close()
        finally:
            stmt.close()

In [ ]:
# Agregar únicamente las fechas y el área 0 que aún no existan.
def insertar_complementos_hechos(conexion, publicaciones, stages):
    for (destino, df, clave), stage in zip(publicaciones, stages):
        columnas = ', '.join(identificador_h(c) for c in df.columns)
        seleccion = ', '.join('s.' + identificador_h(c) for c in df.columns)
        if clave is None:
            continue
        ejecutar_sql_h(conexion, f'INSERT INTO {nombre_sql_h(destino)} ({columnas}) SELECT {seleccion} '
            f'FROM {nombre_sql_h(stage)} s WHERE NOT EXISTS (SELECT 1 FROM {nombre_sql_h(destino)} d '
            f'WHERE d.{identificador_h(clave)} = s.{identificador_h(clave)})')

In [ ]:
# Reemplazar ambos hechos y conciliar las filas antes de confirmar.
def reemplazar_hechos(conexion, publicaciones, stages):
    for destino in ['HechoPlanesTiposBeneficio_ELT', 'HechoHistCondicionesTiposBeneficio_ELT']:
        ejecutar_sql_h(conexion, f'DELETE FROM {nombre_sql_h(destino)}')
    for (destino, df, clave), stage in zip(publicaciones, stages):
        if clave is not None:
            continue
        columnas = ', '.join(identificador_h(c) for c in df.columns)
        ejecutar_sql_h(conexion, f'INSERT INTO {nombre_sql_h(destino)} ({columnas}) SELECT {columnas} FROM {nombre_sql_h(stage)}')
        real = escalar_sql_h(conexion, f'SELECT COUNT(*) FROM {nombre_sql_h(destino)}')
        esperado = df.count()
        if real != esperado:
            raise ValueError(f'{destino}: conciliación fallida ({real} != {esperado}).')

In [92]:
# Confirmar todos los cambios juntos o restaurar el estado anterior.
def publicar_hechos(publicaciones, stages):
    conexion = conexion_jdbc_hechos()
    try:
        validar_motor_hechos(conexion, publicaciones)
        conexion.setAutoCommit(False)
        try:
            insertar_complementos_hechos(conexion, publicaciones, stages)
            reemplazar_hechos(conexion, publicaciones, stages)
            conexion.commit()
            print('Publicación completada: ambos hechos conciliados; reejecutar 5.3 no acumula filas.')
        except Exception:
            conexion.rollback()
            raise
    finally:
        conexion.close()

In [93]:
# Retirar las tablas técnicas de esta ejecución, incluso si hubo un error.
def limpiar_staging_hechos(stages):
    conexion = None
    try:
        conexion = conexion_jdbc_hechos()
        for stage in stages:
            try:
                ejecutar_sql_h(conexion, f'DROP TABLE IF EXISTS {nombre_sql_h(stage)}')
            except Exception:
                print(f'No se pudo eliminar la tabla técnica {stage}; puede retirarse después.')
    except Exception:
        print('No se pudo conectar para limpiar staging. Revisar tablas técnicas de esta ejecución:', stages)
    finally:
        if conexion is not None:
            conexion.close()

**Publicación de los hechos.** La carga reemplaza todo el horizonte extraído de ambos hechos. Las fechas faltantes y el área desconocida se agregan si son necesarias. Si la publicación falla, la transacción revierte sus cambios en las tablas de hechos.


In [ ]:
# Ejecutar la carga por lotes y la publicación de los dos hechos.
# Reemplaza todo el horizonte extraído; no acumula filas al reejecutar, es un borrado y cargado.
token_h = uuid4().hex[:12]
publicaciones_h = [
    ('Fecha_ELT', df_fechas_adicionales_h, 'Fecha'),
    ('AreasDeServicio_ELT', df_area_cero_h, 'IdAreaDeServicio_DWH'),
    ('HechoHistCondicionesTiposBeneficio_ELT', df_hecho_hist_h, None),
    ('HechoPlanesTiposBeneficio_ELT', df_hecho_planes_h, None)]
stages_h = []
try:
    preparar_staging_hechos(publicaciones_h, stages_h, token_h)
    publicar_hechos(publicaciones_h, stages_h)
finally:
    limpiar_staging_hechos(stages_h)

Staging Fecha_ELT: 1 filas verificadas; batchsize=5000.
Staging AreasDeServicio_ELT: 1 filas verificadas; batchsize=5000.
Staging HechoHistCondicionesTiposBeneficio_ELT: 342 filas verificadas; batchsize=5000.
Staging HechoPlanesTiposBeneficio_ELT: 70,752 filas verificadas; batchsize=5000.
Publicación completada: ambos hechos conciliados; reejecutar 5.3 no acumula filas.


# 6. Cierre de PySpark

In [95]:
spark.stop()